In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

train_df = pd.read_csv("/kaggle/sign_mnist_train.csv")
test_df  = pd.read_csv("/kaggle/sign_mnist_test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# 2. Dataset class

class SignLanguageSequenceDataset(Dataset):
    def __init__(self, df):
        self.labels = df["label"].values.astype(np.int64)
        self.images = df.drop(columns=["label"]).values.astype(np.float32)

        # scale pixels to [0,1]
        self.images = self.images / 255.0

        # reshape each image from (784,) to (28, 28)
        self.images = self.images.reshape(-1, 28, 28)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.images[idx], dtype=torch.float32)   # (28, 28)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

train_dataset = SignLanguageSequenceDataset(train_df)
test_dataset  = SignLanguageSequenceDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False)

# 3. LSTM model
class SignLSTM(nn.Module):
    def __init__(self, input_size=28, hidden_size=128, num_layers=2, num_classes=25, dropout=0.3):
        super().__init__()

        # dropout only applies when num_layers > 1
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: (batch_size, seq_len=28, input_size=28)
        lstm_out, (hidden, cell) = self.lstm(x)

        final_hidden = hidden[-1]

        out = self.fc(final_hidden)
        return out

num_classes = int(max(train_df["label"].max(), test_df["label"].max()) + 1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SignLSTM(
    input_size=28,
    hidden_size=128,
    num_layers=2,
    num_classes=num_classes,
    dropout=0.3
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 4. Training loop

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)

        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            running_loss += loss.item() * X.size(0)

            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc, all_labels, all_preds

# 5. Train the model

num_epochs = 15

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion, device)

    print(f"Epoch {epoch+1:02d}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")


test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion, device)

print("\nFinal Test Accuracy:", round(test_acc, 4))
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

Train shape: (27455, 785)
Test shape: (7172, 785)
Epoch 01/15 | Train Loss: 2.3429 | Train Acc: 0.2799 | Test Loss: 1.7268 | Test Acc: 0.4448
Epoch 02/15 | Train Loss: 1.0764 | Train Acc: 0.6569 | Test Loss: 1.0744 | Test Acc: 0.6482
Epoch 03/15 | Train Loss: 0.4800 | Train Acc: 0.8593 | Test Loss: 0.6330 | Test Acc: 0.8002
Epoch 04/15 | Train Loss: 0.2336 | Train Acc: 0.9393 | Test Loss: 0.5927 | Test Acc: 0.8052
Epoch 05/15 | Train Loss: 0.1058 | Train Acc: 0.9783 | Test Loss: 0.5286 | Test Acc: 0.8530
Epoch 06/15 | Train Loss: 0.0629 | Train Acc: 0.9881 | Test Loss: 0.5177 | Test Acc: 0.8532
Epoch 07/15 | Train Loss: 0.0331 | Train Acc: 0.9953 | Test Loss: 0.5007 | Test Acc: 0.8677
Epoch 08/15 | Train Loss: 0.0789 | Train Acc: 0.9781 | Test Loss: 0.4823 | Test Acc: 0.8866
Epoch 09/15 | Train Loss: 0.0104 | Train Acc: 0.9995 | Test Loss: 0.4657 | Test Acc: 0.8862
Epoch 10/15 | Train Loss: 0.0772 | Train Acc: 0.9787 | Test Loss: 0.4829 | Test Acc: 0.8636
Epoch 11/15 | Train Loss: 0.00